In [ ]:
"""
Jupyter Notebook: Evaluate CLIP/BERT Embeddings
================================================
Step-by-step guide to test your embeddings

Usage:
1. Run each cell sequentially
2. Interpret results after each test
3. Make decision at the end
"""

# ============================================================================
# CELL 1: Setup
# ============================================================================

import numpy as np
import pandas as pd
import sys
import os

# Add your project path
sys.path.append('../')

from embedding_quality_evaluator import EmbeddingQualityEvaluator, create_manual_test_cases
from compare_pretrained_vs_finetuned import quick_baseline_test, ModelComparator

import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")


# ============================================================================
# CELL 2: Load Data
# ============================================================================

# Paths (adjust to your structure)
DATA_DIR = "../data/processed"

# Load embeddings
print("Loading embeddings...")
clip_embeddings = np.load(f"{DATA_DIR}/clip_item_embeddings.npy")
variant_ids = np.load(f"{DATA_DIR}/variant_ids.npy", allow_pickle=True)
metadata_df = pd.read_csv(f"{DATA_DIR}/item_features.csv")

# Align data
metadata_df['variant_id'] = metadata_df['variant_id'].astype(str)
metadata_df = metadata_df[metadata_df['variant_id'].isin(variant_ids)]

print(f"✅ Loaded data:")
print(f"  - Embeddings shape: {clip_embeddings.shape}")
print(f"  - Products: {len(metadata_df)}")
print(f"  - Categories: {metadata_df['category_name'].nunique()}")
print(f"\nSample data:")
print(metadata_df[['variant_id', 'product_name', 'category_name', 'price']].head())


# ============================================================================
# CELL 3: Quick Baseline Test (⏱️ ~5 minutes)
# ============================================================================

print("\n" + "="*80)
print("QUICK BASELINE TEST")
print("="*80)
print("This will give you a quick decision on whether to fine-tune or not...")
print()

baseline_score, recommendation = quick_baseline_test(clip_embeddings, metadata_df)

print("\n📋 DECISION:")
print(f"  Baseline Score: {baseline_score:.1f}/100")
print(f"  Recommendation: {recommendation}")
print()

# Interpretation guide
if baseline_score >= 70:
    print("🎉 Great news! Your pretrained model is already good enough.")
    print("   Next steps:")
    print("   1. ✅ Use CLIP pretrained embeddings")
    print("   2. Focus on system optimization (caching, FAISS tuning)")
    print("   3. Collect user interaction data for future improvements")
    
elif baseline_score >= 55:
    print("⚠️  Your pretrained model is acceptable but not great.")
    print("   Two options:")
    print("   1. Use pretrained now, fine-tune later (recommended)")
    print("   2. Fine-tune with synthetic data (10-15% improvement expected)")
    
else:
    print("❌ Your pretrained model quality is too low.")
    print("   Required actions:")
    print("   1. Fine-tune CLIP on your product images")
    print("   2. Use synthetic training data (same category, similar metadata)")
    print("   3. Expected improvement: 15-25%")


# ============================================================================
# CELL 4: Detailed Test - Category Coherence
# ============================================================================

print("\n" + "="*80)
print("DETAILED TEST 1: CATEGORY COHERENCE")
print("="*80)
print("Testing if products in same category are grouped together...")
print()

evaluator = EmbeddingQualityEvaluator(
    embeddings=clip_embeddings,
    metadata_df=metadata_df,
    embedding_type="CLIP"
)

coherence_results = evaluator.test_category_coherence(sample_size=50)

print("\n💡 What this means:")
if coherence_results['coherence_score'] > 1.5:
    print("  ✅ Excellent - Your model strongly understands product categories")
elif coherence_results['coherence_score'] > 1.2:
    print("  ✅ Good - Model understands categories reasonably well")
else:
    print("  ❌ Poor - Model struggles to distinguish categories")
    print("     → Fine-tuning is strongly recommended")


# ============================================================================
# CELL 5: Detailed Test - Nearest Neighbor Quality
# ============================================================================

print("\n" + "="*80)
print("DETAILED TEST 2: NEAREST NEIGHBOR QUALITY")
print("="*80)
print("Testing if similar products are retrieved as neighbors...")
print()

nn_results = evaluator.test_nearest_neighbors(sample_size=50, k=10)

print("\n💡 What this means:")
print(f"  • When searching for a product, {nn_results['category_overlap']:.0%} of top-10")
print(f"    results are from the SAME category")
print(f"  • {nn_results['gender_overlap']:.0%} match gender")
print(f"  • {nn_results['color_overlap']:.0%} match color")
print()

if nn_results['category_overlap'] > 0.7:
    print("  ✅ Excellent - Search results will be very relevant")
elif nn_results['category_overlap'] > 0.5:
    print("  ✅ Good - Search results will be reasonably relevant")
else:
    print("  ❌ Poor - Search results will be mixed quality")


# ============================================================================
# CELL 6: Detailed Test - Attribute Consistency
# ============================================================================

print("\n" + "="*80)
print("DETAILED TEST 3: ATTRIBUTE CONSISTENCY")
print("="*80)
print("Testing if products with same attributes are similar...")
print()

attr_results = evaluator.test_attribute_consistency()

print("\n💡 What this means:")
for attr, metrics in attr_results.items():
    if metrics['ratio'] > 1.3:
        status = "✅ Good"
    elif metrics['ratio'] > 1.1:
        status = "⚠️  Fair"
    else:
        status = "❌ Poor"
    
    print(f"  {status} - {attr}: Products with same {attr} are "
          f"{metrics['ratio']:.2f}x more similar")


# ============================================================================
# CELL 7: Detailed Test - Price Independence
# ============================================================================

print("\n" + "="*80)
print("DETAILED TEST 4: PRICE INDEPENDENCE")
print("="*80)
print("Testing if embeddings are biased by price...")
print()

price_results = evaluator.test_price_similarity_correlation(sample_size=100)

print("\n💡 What this means:")
correlation = abs(price_results['correlation'])

if correlation < 0.1:
    print(f"  ✅ Excellent - Price correlation is very low ({correlation:.3f})")
    print("     → Model captures style/quality, not just price")
elif correlation < 0.3:
    print(f"  ⚠️  Moderate - Some price influence ({correlation:.3f})")
    print("     → Acceptable for most use cases")
else:
    print(f"  ❌ High - Strong price bias ({correlation:.3f})")
    print("     → Model might group by price rather than similarity")


# ============================================================================
# CELL 8: Create and Test Manual Cases (OPTIONAL)
# ============================================================================

print("\n" + "="*80)
print("MANUAL TEST CASES (OPTIONAL)")
print("="*80)
print("You can create specific test cases based on your domain knowledge...")
print()

# Example: Create manual test cases
# You should customize these based on your actual products

def create_custom_test_cases(metadata_df):
    """
    Create domain-specific test cases
    Adjust this based on your actual product data
    """
    test_cases = []
    
    # Example 1: T-shirts should be similar to each other
    tshirts = metadata_df[
        metadata_df['product_name'].str.contains('T-shirt|tshirt', case=False, na=False)
    ]
    
    if len(tshirts) >= 5:
        anchor = tshirts.iloc[0]
        similar = tshirts.iloc[1:4]['variant_id'].tolist()
        
        # Dissimilar: shoes or accessories
        dissimilar = metadata_df[
            metadata_df['category_name'].str.contains('Shoe|Accessories', case=False, na=False)
        ].head(2)['variant_id'].tolist()
        
        if dissimilar:
            test_cases.append({
                'anchor': anchor['variant_id'],
                'similar': similar,
                'dissimilar': dissimilar,
                'description': 'T-shirts vs Shoes/Accessories'
            })
    
    # Add more test cases based on your domain...
    
    return test_cases

# Create and test
manual_cases = create_custom_test_cases(metadata_df)

if manual_cases:
    manual_results = evaluator.test_manual_cases(manual_cases)
    print(f"\n✅ Manual test accuracy: {manual_results['accuracy']:.1%}")
else:
    print("\n⚠️  No manual test cases created")
    print("   → Customize create_custom_test_cases() for your domain")


# ============================================================================
# CELL 9: Comprehensive Report
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION REPORT")
print("="*80)

# Run all tests
full_results = evaluator.run_all_tests(manual_test_cases=manual_cases if manual_cases else None)

# Save results
import json
output_path = f"{DATA_DIR}/evaluation_results.json"
with open(output_path, 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    def convert_to_serializable(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    
    json.dump(full_results, f, indent=2, default=convert_to_serializable)

print(f"\n✅ Full results saved to: {output_path}")


# ============================================================================
# CELL 10: Final Decision & Recommendations
# ============================================================================

print("\n" + "="*80)
print("FINAL DECISION & RECOMMENDATIONS")
print("="*80)

overall_score = full_results['overall_score']

print(f"\n🎯 OVERALL SCORE: {overall_score:.1f}/100\n")

if overall_score >= 80:
    print("="*80)
    print("✅ DECISION: USE PRETRAINED MODEL AS-IS")
    print("="*80)
    print("\nYour pretrained CLIP embeddings are excellent quality!")
    print("\n📋 Next Steps:")
    print("  1. Deploy current embeddings to production")
    print("  2. Focus on system optimization:")
    print("     - Optimize FAISS index (IVF, PQ parameters)")
    print("     - Implement caching strategy")
    print("     - A/B test recommendation UI")
    print("  3. Start collecting user interaction data")
    print("  4. Revisit fine-tuning after 3-6 months with real data")
    print("\n💰 Estimated time saved: 1-2 weeks")
    print("="*80)

elif overall_score >= 65:
    print("="*80)
    print("⚠️  DECISION: USE PRETRAINED, PLAN FINE-TUNING")
    print("="*80)
    print("\nYour pretrained model is acceptable but has room for improvement.")
    print("\n📋 Recommended Path:")
    print("  Option A (Recommended):")
    print("    1. Deploy pretrained model now")
    print("    2. Collect 2-3 months of interaction data")
    print("    3. Fine-tune with real user behavior data")
    print("    4. Expected improvement: 15-20%")
    print("\n  Option B (If quality is critical):")
    print("    1. Fine-tune now with synthetic data")
    print("    2. Expected improvement: 8-12%")
    print("    3. Time investment: 1 week")
    print("    4. Fine-tune again later with real data")
    print("="*80)

else:
    print("="*80)
    print("❌ DECISION: MUST FINE-TUNE BEFORE PRODUCTION")
    print("="*80)
    print("\nYour pretrained model quality is insufficient for production.")
    print("\n📋 Required Actions:")
    print("  1. Fine-tune CLIP on your product images")
    print("  2. Create training data:")
    print("     - Positive pairs: Same category + similar metadata")
    print("     - Negative pairs: Different categories")
    print("     - Minimum 5,000 pairs (automatic generation)")
    print("  3. Fine-tune for 3-5 epochs")
    print("  4. Re-evaluate (expect 15-25% improvement)")
    print("\n⏱️  Estimated time: 1 week")
    print("  - Data preparation: 1 day")
    print("  - Training: 1-2 days (with your RTX 4050)")
    print("  - Evaluation & iteration: 2-3 days")
    print("="*80)

print("\n\n💡 ADDITIONAL INSIGHTS:")

# Category coherence interpretation
coherence = full_results['category_coherence']['coherence_score']
print(f"\n1. Category Understanding: {coherence:.2f}x")
if coherence < 1.2:
    print("   ⚠️  Model struggles with your product categories")
    print("   → Consider fine-tuning with category labels")

# NN quality interpretation
nn_overlap = full_results['nearest_neighbors']['category_overlap']
print(f"\n2. Search Quality: {nn_overlap:.1%} category overlap")
if nn_overlap < 0.6:
    print("   ⚠️  Search results may be mixed quality")
    print("   → Add category filtering in your ranking layer")

# Price bias interpretation
price_corr = abs(full_results['price_correlation']['correlation'])
print(f"\n3. Price Bias: {price_corr:.3f} correlation")
if price_corr > 0.3:
    print("   ⚠️  Strong price bias detected")
    print("   → Consider adding price-independent features")

print("\n" + "="*80)
print("📊 Evaluation Complete!")
print("="*80)


# ============================================================================
# CELL 11: Visualize Sample Recommendations
# ============================================================================

print("\n" + "="*80)
print("SAMPLE RECOMMENDATIONS VISUALIZATION")
print("="*80)
print("Let's see actual recommendation quality for a sample product...")
print()

from sklearn.metrics.pairwise import cosine_similarity

# Pick a random product
sample_idx = np.random.randint(0, len(metadata_df))
sample_product = metadata_df.iloc[sample_idx]
sample_embedding = clip_embeddings[sample_idx]

print(f"🔍 Query Product:")
print(f"  Name: {sample_product['product_name']}")
print(f"  Category: {sample_product['category_name']}")
print(f"  Price: ${sample_product['price']:.2f}")
print()

# Find top 10 similar products
similarities = cosine_similarity([sample_embedding], clip_embeddings)[0]
top_10_indices = np.argsort(similarities)[::-1][1:11]  # Exclude self

print(f"📋 Top 10 Recommendations:")
print("-" * 80)

for rank, idx in enumerate(top_10_indices, 1):
    rec_product = metadata_df.iloc[idx]
    similarity = similarities[idx]
    
    # Check if same category
    same_cat = "✓" if rec_product['category_name'] == sample_product['category_name'] else "✗"
    
    print(f"{rank:2d}. [{same_cat}] {rec_product['product_name'][:50]:50s} "
          f"({rec_product['category_name'][:20]:20s}) "
          f"${rec_product['price']:6.2f} | sim={similarity:.3f}")

# Calculate quality metrics
same_category_count = sum(
    1 for idx in top_10_indices 
    if metadata_df.iloc[idx]['category_name'] == sample_product['category_name']
)

print("-" * 80)
print(f"Quality: {same_category_count}/10 recommendations from same category")
print()

if same_category_count >= 7:
    print("✅ Excellent recommendation quality for this product!")
elif same_category_count >= 5:
    print("⚠️  Fair recommendation quality")
else:
    print("❌ Poor recommendation quality - fine-tuning recommended")

print("\n💡 Tip: Run this cell multiple times to check different products")


# ============================================================================
# CELL 12: Export Decision Summary
# ============================================================================

# Create decision summary document
summary = f"""
EMBEDDING EVALUATION SUMMARY
============================
Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

OVERALL SCORE: {overall_score:.1f}/100

KEY METRICS:
- Category Coherence: {coherence:.2f}x
- NN Category Overlap: {nn_overlap:.1%}
- Price Correlation: {price_corr:.3f}

DECISION: {"USE PRETRAINED" if overall_score >= 70 else "FINE-TUNE" if overall_score < 55 else "CONSIDER FINE-TUNING"}

REASONING:
"""

if overall_score >= 70:
    summary += "Pretrained model quality is excellent. No immediate fine-tuning needed.\n"
elif overall_score >= 55:
    summary += "Pretrained model is acceptable. Fine-tuning can wait until interaction data is available.\n"
else:
    summary += "Pretrained model quality is insufficient. Fine-tuning is required before production.\n"

# Save summary
summary_path = f"{DATA_DIR}/evaluation_summary.txt"
with open(summary_path, 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✅ Summary saved to: {summary_path}")

print("\n" + "="*80)
print("🎉 EVALUATION WORKFLOW COMPLETE!")
print("="*80)
print("\nYou now have all the data needed to make an informed decision.")
print("Check the following files:")
print(f"  1. {DATA_DIR}/evaluation_results.json (detailed metrics)")
print(f"  2. {DATA_DIR}/evaluation_summary.txt (decision summary)")
print("\nNext: Follow the recommendations above to proceed with your project!")

ImportError: cannot import name '_imaging' from 'PIL' (/usr/lib/python3/dist-packages/PIL/__init__.py)

In [4]:
pip install numpy pandas scikit-learn tqdm matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 137.1 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 3.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 4.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 5.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 5.7 MB/s eta 0:00